# 05 - Adversarial Attacks

Evaluate the vulnerability of the Tier 2 DNN detector to gradient-based adversarial attacks.

| Attack | Method | Key Param |
|--------|--------|-----------|
| **FGSM** | Single-step gradient sign | epsilon |
| **PGD** | Multi-step projected gradient descent | epsilon, alpha, iterations |

**Sections:**
1. Prepare model and data
2. Run FGSM attack
3. Run PGD attack
4. Evaluate attack success rates
5. Visualize perturbations
6. Epsilon sensitivity analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F

from src.preprocessing.data_loader import DataLoader, CATEGORY_MAP
from src.preprocessing.preprocessor import DataPreprocessor
from src.adversarial_attacks.fgsm import fgsm_attack
from src.adversarial_attacks.pgd import pgd_attack
from src.adversarial_attacks.attack_utils import PyTorchDNN, evaluate_attack
from src.utils.config import load_config

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Imports loaded.')

## 1. Prepare Model and Data

In [ ]:
config = load_config('../config/config.yaml')
loader = DataLoader()

try:
    df = loader.load(config)
except FileNotFoundError:
    print('Using synthetic data.')
    df = loader.generate_synthetic(n_samples=5000)

preprocessor = DataPreprocessor(config)
data = preprocessor.run_pipeline(df, label_type='multiclass')

n_features = data['n_features']
n_classes = data['n_classes']

# Create PyTorch DNN model (mirrors the Keras DNN architecture)
pt_model = PyTorchDNN(n_features, n_classes)

# Quick training so the model has learned patterns to attack
optimizer = torch.optim.Adam(pt_model.parameters(), lr=0.001)
X_train_t = torch.FloatTensor(data['X_train'])
y_train_t = torch.LongTensor(data['y_train'])

print('Training PyTorch DNN for attack experiments...')
pt_model.train()
for epoch in range(10):
    for i in range(0, len(X_train_t), 256):
        xb = X_train_t[i:i+256]
        yb = y_train_t[i:i+256]
        optimizer.zero_grad()
        loss = F.cross_entropy(pt_model(xb), yb)
        loss.backward()
        optimizer.step()

# Evaluate clean accuracy
pt_model.eval()
X_test_t = torch.FloatTensor(data['X_test'])
y_test_t = torch.LongTensor(data['y_test'])
with torch.no_grad():
    clean_preds = pt_model(X_test_t).argmax(dim=1).numpy()
clean_acc = (clean_preds == data['y_test']).mean()
print(f'\nClean test accuracy: {clean_acc:.4f}')

## 2. FGSM Attack

**Fast Gradient Sign Method** (Goodfellow et al., 2014):

$$x_{adv} = x + \epsilon \cdot \text{sign}(\nabla_x \mathcal{L}(\theta, x, y))$$

A single-step attack that perturbs inputs in the direction of the loss gradient.

In [ ]:
# Use a subset for faster computation
N_ATTACK = min(500, len(data['X_test']))
x_sample = torch.FloatTensor(data['X_test'][:N_ATTACK])
y_sample = torch.LongTensor(data['y_test'][:N_ATTACK])

epsilon_fgsm = config['adversarial_attacks']['fgsm']['epsilon']
print(f'Running FGSM with epsilon={epsilon_fgsm}...')

x_adv_fgsm = fgsm_attack(pt_model, x_sample, y_sample, epsilon=epsilon_fgsm)

fgsm_results = evaluate_attack(pt_model, x_sample, x_adv_fgsm, y_sample)

print(f'\nFGSM Results:')
print(f'  Attack Success Rate : {fgsm_results["attack_success_rate"]:.2f}%')
print(f'  Samples Fooled      : {fgsm_results["samples_fooled"]}/{fgsm_results["samples_attacked"]}')
print(f'  Avg L2 Perturbation : {fgsm_results["avg_l2_perturbation"]:.4f}')
print(f'  Avg Linf Perturbation: {fgsm_results["avg_linf_perturbation"]:.4f}')

## 3. PGD Attack

**Projected Gradient Descent** (Madry et al., 2018):

$$x_{t+1} = \Pi_{x+\mathcal{S}}\big(x_t + \alpha \cdot \text{sign}(\nabla_x \mathcal{L})\big)$$

An iterative version of FGSM that takes multiple small steps, projected back
into the epsilon-ball around the original input.

In [ ]:
pgd_config = config['adversarial_attacks']['pgd']
epsilon_pgd = pgd_config['epsilon']
alpha_pgd = pgd_config['alpha']
iters_pgd = min(pgd_config['num_iterations'], 20)  # cap for speed

print(f'Running PGD with epsilon={epsilon_pgd}, alpha={alpha_pgd}, iterations={iters_pgd}...')

x_adv_pgd = pgd_attack(
    pt_model, x_sample, y_sample,
    epsilon=epsilon_pgd, alpha=alpha_pgd,
    num_iterations=iters_pgd, random_start=True
)

pgd_results = evaluate_attack(pt_model, x_sample, x_adv_pgd, y_sample)

print(f'\nPGD Results:')
print(f'  Attack Success Rate : {pgd_results["attack_success_rate"]:.2f}%')
print(f'  Samples Fooled      : {pgd_results["samples_fooled"]}/{pgd_results["samples_attacked"]}')
print(f'  Avg L2 Perturbation : {pgd_results["avg_l2_perturbation"]:.4f}')
print(f'  Avg Linf Perturbation: {pgd_results["avg_linf_perturbation"]:.4f}')

## 4. Attack Comparison

In [ ]:
comparison = pd.DataFrame([
    {'Attack': 'FGSM', **fgsm_results},
    {'Attack': 'PGD', **pgd_results}
])

print('Attack Comparison:')
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

attacks = ['FGSM', 'PGD']
colors = ['#e74c3c', '#3498db']

# Success rate
axes[0].bar(attacks, [fgsm_results['attack_success_rate'], pgd_results['attack_success_rate']],
            color=colors)
axes[0].set_title('Attack Success Rate (%)')
axes[0].set_ylabel('%')
for i, v in enumerate([fgsm_results['attack_success_rate'], pgd_results['attack_success_rate']]):
    axes[0].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

# L2 perturbation
axes[1].bar(attacks, [fgsm_results['avg_l2_perturbation'], pgd_results['avg_l2_perturbation']],
            color=colors)
axes[1].set_title('Avg L2 Perturbation')

# Linf perturbation
axes[2].bar(attacks, [fgsm_results['avg_linf_perturbation'], pgd_results['avg_linf_perturbation']],
            color=colors)
axes[2].set_title('Avg Linf Perturbation')

plt.suptitle('FGSM vs PGD Attack Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Visualize Perturbations

Show how adversarial perturbations modify feature values.

In [ ]:
# Perturbation magnitude per feature
fgsm_perturbation = (x_adv_fgsm - x_sample).abs().mean(dim=0).numpy()
pgd_perturbation = (x_adv_pgd - x_sample).abs().mean(dim=0).numpy()

feature_names = data.get('feature_names', [f'F{i}' for i in range(n_features)])

fig, ax = plt.subplots(figsize=(14, 5))
x_pos = np.arange(n_features)
width = 0.35
ax.bar(x_pos - width/2, fgsm_perturbation, width, label='FGSM', color='#e74c3c', alpha=0.8)
ax.bar(x_pos + width/2, pgd_perturbation, width, label='PGD', color='#3498db', alpha=0.8)
ax.set_xlabel('Feature Index')
ax.set_ylabel('Mean |Perturbation|')
ax.set_title('Average Perturbation Magnitude per Feature')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize a single sample: clean vs adversarial
idx = 0  # first sample
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(n_features), x_sample[idx].numpy(), alpha=0.7, label='Clean', color='steelblue')
axes[0].bar(range(n_features), x_adv_fgsm[idx].numpy(), alpha=0.5, label='FGSM Adv', color='red')
axes[0].set_title('Sample 0: Clean vs FGSM Adversarial')
axes[0].set_xlabel('Feature Index')
axes[0].legend()

axes[1].bar(range(n_features), x_sample[idx].numpy(), alpha=0.7, label='Clean', color='steelblue')
axes[1].bar(range(n_features), x_adv_pgd[idx].numpy(), alpha=0.5, label='PGD Adv', color='orange')
axes[1].set_title('Sample 0: Clean vs PGD Adversarial')
axes[1].set_xlabel('Feature Index')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Epsilon Sensitivity Analysis

Test how model accuracy degrades as the perturbation budget (epsilon) increases.

In [ ]:
epsilons = [0.0, 0.01, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3]
fgsm_accs = []
pgd_accs = []

for eps in epsilons:
    if eps == 0:
        fgsm_accs.append(clean_acc)
        pgd_accs.append(clean_acc)
        continue

    # FGSM
    x_f = fgsm_attack(pt_model, x_sample, y_sample, epsilon=eps)
    with torch.no_grad():
        pf = pt_model(x_f).argmax(dim=1).numpy()
    fgsm_accs.append((pf == y_sample.numpy()).mean())

    # PGD
    x_p = pgd_attack(pt_model, x_sample, y_sample, epsilon=eps, alpha=eps/4, num_iterations=10)
    with torch.no_grad():
        pp = pt_model(x_p).argmax(dim=1).numpy()
    pgd_accs.append((pp == y_sample.numpy()).mean())

print('Epsilon sensitivity results computed.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epsilons, fgsm_accs, marker='o', label='FGSM', color='#e74c3c', linewidth=2)
ax.plot(epsilons, pgd_accs, marker='s', label='PGD', color='#3498db', linewidth=2)
ax.set_xlabel('Epsilon (Perturbation Budget)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Model Accuracy vs Adversarial Perturbation Strength', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
eps_df = pd.DataFrame({
    'Epsilon': epsilons,
    'FGSM Accuracy': [f'{a:.4f}' for a in fgsm_accs],
    'PGD Accuracy': [f'{a:.4f}' for a in pgd_accs],
})
print(eps_df.to_string(index=False))

## Summary

- Both FGSM and PGD attacks significantly degrade model accuracy.
- PGD (iterative) is generally stronger than FGSM (single-step) for the same epsilon.
- Accuracy drops sharply with increasing epsilon, demonstrating the need for adversarial defenses.
- Feature perturbations are small in magnitude but enough to fool the classifier.

Next: `06_defense_evaluation.ipynb` -- evaluate defenses against these attacks.